<a href="https://colab.research.google.com/github/dev-aishwarya-hub/daytona/blob/main/DOMAIN_SPECIFIC_CHATBOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# INSTALL LIBRARIES

In [19]:

!pip install -q -U \
     Sentence-transformers==3.0.1 \
     langchain==0.3.19 \
     langchain-groq==0.2.4 \
     langchain-community==0.3.18 \
     langchain-huggingface==0.1.2 \
     einops==0.8.1 \
     faiss_cpu==1.10.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.3 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.3.84 which is incompatible.
langchain-classic 1.0.3 requires langchain-text-splitters<2.0.0,>=1.1.1, but you have langchain-text-splitters 0.3.11 which is incompatible.
langgraph-prebuilt 1.0.9 requires langchain-core>=1.0.0, but you have langchain-core 0.3.84 which is incompatible.


# IMPORT LIBRARIES

In [3]:
# Import Libraries
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate


In [4]:

import getpass
import os

GROQ API KEY

In [5]:
os.environ["GROQ_API_KEY"] = getpass.getpass()

··········


HUGGING FACE TOKEN

In [7]:

os.environ["HF_TOKEN"] = getpass.getpass()

··········


In [8]:

# Helper function for printing docs
def pretty_print_docs(docs):
    # Iterate through each document and format the output
    for i, d in enumerate(docs):
        print(f"{'-' * 50}\nDocument {i + 1}:")
        print(f"Content:\n{d.page_content}\n")
        print("Metadata:")
        for key, value in d.metadata.items():
            print(f"  {key}: {value}")
    print(f"{'-' * 50}")  # Final separator for clarity


In [9]:
!pip install -qU langchain-community pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 0.1.2 requires langchain-core<0.4.0,>=0.3.15, but you have langchain-core 1.2.29 which is incompatible.
langchain-groq 0.2.4 requires langchain-core<0.4.0,>=0.3.33, but you have langchain-core 1.2.29 which is incompatible.
langchain 0.3.19 requires langchain-core<1.0.0,>=0.3.35, but you have langchain-core 1.2.29 which is incompatible.
langchain 0.3.19 requires langchain-text-splitters<1.0.0,>=0.3.6, but you have langchain-text-splitters 1.1.1 which is incompatible.


# Basic RAG in Action

In [10]:
# Import necessary libraries
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [11]:
# Load the PDF
file_path = "/content/Diagnosis.pdf"




In [12]:
loader = PyPDFLoader(file_path)


In [13]:
documents = loader.load()

In [14]:
# Split documents into chunks of 500 characters with 100 characters overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(documents)


In [15]:
# Add unique IDs to each text chunk
for idx, text in enumerate(texts):
    text.metadata["id"] = idx


In [16]:
print(documents)

[Document(metadata={'producer': 'Acrobat Distiller 7.0 for Macintosh', 'creator': 'QuarkXPress: pictwpstops filter 1.0', 'creationdate': '2011-06-16T14:16:22+05:30', 'author': 'Scott D. C. Stern', 'codemantra, llc': 'http://www.codemantra.com', 'universal pdf': 'The process that creates this PDF constitutes a trade secret of codeMantra, LLC and is protected by the copyright laws of the United States', 'moddate': '2015-02-09T01:12:35+02:00', 'title': 'Symptom to Diagnosis', 'source': '/content/Diagnosis.pdf', 'total_pages': 508, 'page': 0, 'page_label': 'C'}, page_content=''), Document(metadata={'producer': 'Acrobat Distiller 7.0 for Macintosh', 'creator': 'QuarkXPress: pictwpstops filter 1.0', 'creationdate': '2011-06-16T14:16:22+05:30', 'author': 'Scott D. C. Stern', 'codemantra, llc': 'http://www.codemantra.com', 'universal pdf': 'The process that creates this PDF constitutes a trade secret of codeMantra, LLC and is protected by the copyright laws of the United States', 'moddate': '2

# EMBEDDINGS

In [17]:
# Create embeddings for the text chunks
embedding = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5", model_kwargs = {'trust_remote_code': True})


In [18]:

# Initialize a FAISS (Vector Store) retriever with the text embeddings
retriever = FAISS.from_documents(texts, embedding).as_retriever(search_kwargs={"k": 2})

# RAG CHAINING

In [ ]:
# Import the RetrievalQA chain for question-answering tasks
from langchain.chains import RetrievalQA


In [ ]:
# Define a query and retrieve relevant documents
query = "What is malnutrition?"

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2
)


In [ ]:
# Create a RetrievalQA chain using the language model and the retriever
chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)


In [ ]:
from sentence_transformers import SentenceTransformer, util



def chatbot_response(query, threshold=0.5):
    # Encode the query
    query_embedding = model.encode(query, convert_to_tensor=True)

    # Compute similarity with knowledge base
    similarities = util.cos_sim(query_embedding, doc_embeddings)[0]
    max_score = float(similarities.max())

    if max_score < threshold:
        return "I don’t have relevant knowledge in this respect."
    else:
        best_match_idx = int(similarities.argmax())
        return f"Relevant info: {documents[best_match_idx]}"



In [ ]:
# Invoke the chain with a specific query to get a response
reponse = chain.invoke(query)


In [ ]:
# Print the result of the response
print(reponse["result"])

# GRADIO FOR UI

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

def chat_fn(message, history):
    response = chain.invoke({"query": message})
    return response["result"]

theme = gr.themes.Soft(
    primary_hue="blue",      # main color
    secondary_hue="purple",  # accent
    neutral_hue="gray"
)

demo = gr.ChatInterface(
    fn=chat_fn,
    theme=theme
)

demo.launch(share=True)